# ***car_dct System***
1. Enter in ***XL Tables***
2. Move to ***Jupyter dictionaries***
3. Result is all **"cause data"** and **"results data"** in one **"car_dct"**

# ***car_dct System*** Background

*Click This to see copilot response:* _>>_ Description of "car_dct_"
https://copilot.microsoft.com/shares/n9o1yfsbarWzGzWLXmb41

*Click This to see copilot response:* _>>_ DESIGN of "car_dct_"
https://copilot.microsoft.com/shares/64zdAPPgXqmCRA4GzKNwx

# Implementation

## 📘 Imports

In [3]:
import pandas as pd
from pathlib import Path


## 📘 1. Load Excel tables into raw dictionaries

In [88]:
def load_excel_table(path, name):
    df = pd.read_excel(path)
    return df, {
        "name": f"{name}_raw_dct",
        "class": "raw",
        "source": {
            "file": str(path),
            "sheet": None,
            "import_time": None,
            "manual_notes": "Copied manually from Excel to WSL/data"
        },
        "dependencies": [],
        "description": f"Raw import of {name}"
    }

raw_tables = {}
metadata_dct = {}         
# JL_2 is your project directory
data_dir = Path("/home/ratlabs/JL_2/data")

for name in ["ib_97"]:
    df, meta = load_excel_table(data_dir / f"{name}.xlsx", name)
    raw_tables[f"{name}_raw_dct"] = df
    metadata_dct[f"{name}_raw_dct"] = meta


In [89]:
# vefify raw_tables

## 📘 2. Build dtv spine (dtv_dct)

In [90]:
def build_dtv_spine(start_dtv, end_dtv):
    start = pd.to_datetime(str(start_dtv), format="%Y%m%d")
    end   = pd.to_datetime(str(end_dtv),   format="%Y%m%d")

    dates = pd.date_range(start=start, end=end, freq="D")
    dtv_list = dates.strftime("%Y%m%d").astype(int)

    df = pd.DataFrame({
        "dtv": dtv_list,
        "yr/mo/day": dates.strftime("%Y/%m/%d"),
        "notes": ""
    })
    return df

dtv_dct = build_dtv_spine(20250101, 20250901)                    # sets start and stop



## 📘 3. Row filtering

In [75]:
crt_lst_dct = {
    "ib77_testA": [("account", "A77"), ("time_of_day", "morning")],
    "ib97_testB": [("account", "A97")],
    "dat_mnl_testC": [("subject", "manual")]
}


In [76]:
def apply_row_filter(df, criteria):
    filtered = df.copy()
    for col, val in criteria:
        filtered = filtered[filtered[col] == val]
    return filtered


In [77]:
filtered_tables = {}

for raw_name, df in raw_tables.items():
    test_name = raw_name.replace("_raw_dct", "") + "_testA"  # example
    criteria = crt_lst_dct.get(test_name, [])
    flt_df = apply_row_filter(df, criteria)

    flt_name = raw_name.replace("_raw_dct", "_flt_dct")
    filtered_tables[flt_name] = flt_df

    metadata_dct[flt_name] = {
        "name": flt_name,
        "class": "filtered",
        "dependencies": [raw_name],
        "criteria": criteria,
        "description": f"Row-filtered version of {raw_name}"
    }


## 📘 4. Column filtering

In [91]:
# This the list out of all the data fields that are in the  run
clc_lst_dct = {
    #"ib77_testA": ["dtv", "PhaseAngle", "ECWRatio"],
    "ib97_testB": ["dtv", "MuscleMass"],
    #"dat_mnl_testC": ["dtv", "ManualEntry"]
}


In [79]:
def apply_column_filter(df, columns):
    return df[columns].copy()


In [80]:
selected_tables = {}

for flt_name, df in filtered_tables.items():
    test_name = flt_name.replace("_flt_dct", "") + "_testA"
    columns = clc_lst_dct.get(test_name, ["dtv"])

    sel_df = apply_column_filter(df, columns)
    sel_name = flt_name.replace("_flt_dct", "_clc_dct")
    selected_tables[sel_name] = sel_df

    metadata_dct[sel_name] = {
        "name": sel_name,
        "class": "selected",
        "dependencies": [flt_name],
        "columns": columns,
        "description": f"Column-selected version of {flt_name}"
    }


In [93]:
# verify 
clc_lst_dct

{'ib97_testB': ['dtv', 'MuscleMass']}

## 📘 5. Align each selected dictionary to dtv spine

In [95]:
def align_to_dtv_spine(dtv_df, df):
    return dtv_df.merge(df, on="dtv", how="left")


In [96]:
aligned_tables = {}

for sel_name, df in selected_tables.items():
    aln_df = align_to_dtv_spine(dtv_dct, df)
    aln_name = sel_name.replace("_clc_dct", "_aln_dct")
    aligned_tables[aln_name] = aln_df

    metadata_dct[aln_name] = {
        "name": aln_name,
        "class": "aligned",
        "dependencies": ["dtv_dct", sel_name],
        "description": f"Aligned to dtv spine"
    }


## 📘 6. Merge into master car_dct

In [97]:
def merge_feature_blocks(dtv_df, aligned_dicts):
    car = dtv_df.copy()
    for name, df in aligned_dicts.items():
        cols_to_add = [c for c in df.columns if c not in car.columns]
        car = car.merge(df[["dtv"] + cols_to_add], on="dtv", how="left")
    return car

car_dct = merge_feature_blocks(dtv_dct, aligned_tables)

metadata_dct["car_dct"] = {
    "name": "car_dct",
    "class": "merged",
    "dependencies": list(aligned_tables.keys()),
    "description": "Master cause-and-response dictionary"
}


In [85]:
def merge_feature_blocks(dtv_df, aligned_dicts):
    car = dtv_df.copy()
    for name, df in aligned_dicts.items():
        cols_to_add = [c for c in df.columns if c not in car.columns]
        car = car.merge(df[["dtv"] + cols_to_add], on="dtv", how="left")
    return car

car_dct = merge_feature_blocks(dtv_dct, aligned_tables)

metadata_dct["car_dct"] = {
    "name": "car_dct",
    "class": "merged",
    "dependencies": list(aligned_tables.keys()),
    "description": "Master cause-and-response dictionary"
}


In [87]:
#verify car_dct

In [99]:
#verify 
metadata_dct

{'ib_97_raw_dct': {'name': 'ib_97_raw_dct',
  'class': 'raw',
  'source': {'file': '/home/ratlabs/JL_2/data/ib_97.xlsx',
   'sheet': None,
   'import_time': None,
   'manual_notes': 'Copied manually from Excel to WSL/data'},
  'dependencies': [],
  'description': 'Raw import of ib_97'},
 'ib_97_aln_dct': {'name': 'ib_97_aln_dct',
  'class': 'aligned',
  'dependencies': ['dtv_dct', 'ib_97_clc_dct'],
  'description': 'Aligned to dtv spine'},
 'car_dct': {'name': 'car_dct',
  'class': 'merged',
  'dependencies': ['ib_97_aln_dct'],
  'description': 'Master cause-and-response dictionary'}}

## ⭐ You now have a complete pipeline skeleton

### 1. This code gives you:

raw import

row filtering

column filtering

dtv alignment

merging

metadata tracking

All in clean, modular blocks.